# Cloud Cover Analysis for Mission Planning

Cloud cover is a primary constraint for airborne remote sensing campaigns.
HyPlan's `clouds` module uses **Google Earth Engine** to retrieve historical
MODIS cloud fraction data and simulate realistic mission visit schedules.

We cover:

1. Google Earth Engine setup and authentication
2. Retrieving MODIS cloud fraction for study areas
3. Simulating mission visits with cloud, rest, and weekend constraints
4. Visualizing cloud fraction heatmaps with visit markers
5. Interpreting results for campaign planning

> **Prerequisite:** This notebook requires a Google Earth Engine account.
> Sign up at [earthengine.google.com](https://earthengine.google.com/)
> and authenticate before running.

In [ ]:
import ee

# Authenticate and initialize Earth Engine
# If running for the first time, this will open a browser for authentication.
try:
    ee.Initialize()
    print("Earth Engine initialized successfully.")
except Exception:
    ee.Authenticate()
    ee.Initialize()
    print("Earth Engine authenticated and initialized.")

In [ ]:
from hyplan.clouds import (
    create_cloud_data_array_with_limit,
    simulate_visits,
    plot_yearly_cloud_fraction_heatmaps_with_visits,
)

## 1. Study Area Definition

Cloud data is retrieved for polygons defined in a GeoJSON file. Each
polygon represents a study area (flight box) that needs to be imaged.
The GeoJSON must have a `Name` column identifying each polygon.

We use a sample file from the examples directory.

In [ ]:
import geopandas as gpd

polygon_file = "../examples/data/hyspiri.geojson"

# Preview the study areas
gdf = gpd.read_file(polygon_file)
print(f"Study areas: {len(gdf)} polygons")
print(gdf[["Name"]].to_string(index=False))

gdf.plot(figsize=(8, 5), alpha=0.4, edgecolor="black")

## 2. Retrieve MODIS Cloud Fraction

`create_cloud_data_array_with_limit` queries Google Earth Engine for
daily MODIS cloud fraction over each polygon across multiple years.
This gives the historical cloud climatology needed to estimate how
many clear days are available during a campaign window.

Parameters:
- `polygon_file`: GeoJSON with study-area polygons
- `year_start`, `year_stop`: Range of years to query
- `day_start`, `day_stop`: Day-of-year window (e.g., 1-75 for Jan-Mar)

In [ ]:
# Retrieve cloud data for January through mid-March, 2003-2006
cloud_data_df = create_cloud_data_array_with_limit(
    polygon_file=polygon_file,
    year_start=2003,
    year_stop=2006,
    day_start=1,
    day_stop=75,
)

print(f"Cloud data shape: {cloud_data_df.shape}")
print(f"Columns: {list(cloud_data_df.columns)}")
cloud_data_df.head(10)

## 3. Simulate Mission Visits

`simulate_visits` takes the cloud fraction data and simulates a
realistic mission schedule. On each day, it checks whether each
study area is below the cloud fraction threshold. If so, the area
is "visited" (imaged). The simulation respects:

- **Cloud fraction threshold** — maximum cloudiness for a flyable day
- **Rest day schedule** — mandatory rest after N consecutive flight days
- **Weekend exclusion** — optionally skip weekends

In [ ]:
# Simulate visits for February-March window
day_start = 30
day_stop = 74

visit_days_df, visit_tracker, rest_days = simulate_visits(
    cloud_data_df,
    day_start=day_start,
    day_stop=day_stop,
    year_start=2003,
    year_stop=2006,
    cloud_fraction_threshold=0.25,  # Max 25% cloud cover
    rest_day_threshold=2,           # Rest after 2 consecutive flight days
    exclude_weekends=True,          # No flights on weekends
    debug=True,
)

print(f"\nVisit results:")
print(f"  Shape: {visit_days_df.shape}")
visit_days_df.head()

## 4. Visualize Cloud Fraction Heatmaps

The heatmap shows daily cloud fraction for each study area across the
campaign window. Visit days are marked, along with rest days and
weekends, giving a complete picture of mission scheduling constraints.

In [ ]:
plot_yearly_cloud_fraction_heatmaps_with_visits(
    cloud_data_df,
    visit_tracker,
    rest_days,
    cloud_fraction_threshold=0.25,
    exclude_weekends=True,
    day_start=day_start,
    day_stop=day_stop,
)

## 5. Campaign Planning Insights

Use the simulation results to answer key planning questions:

- How many flyable days per study area?
- What cloud threshold gives adequate coverage?
- How long should the campaign window be?

In [ ]:
# Compare different cloud thresholds
thresholds = [0.15, 0.25, 0.35, 0.50]

print(f"Flyable days by cloud fraction threshold (DOY {day_start}-{day_stop}):\n")

for thresh in thresholds:
    vd, vt, rd = simulate_visits(
        cloud_data_df,
        day_start=day_start,
        day_stop=day_stop,
        year_start=2003,
        year_stop=2006,
        cloud_fraction_threshold=thresh,
        rest_day_threshold=2,
        exclude_weekends=True,
        debug=False,
    )
    total_visits = vd.sum().sum() if len(vd) > 0 else 0
    print(f"  Threshold {thresh:.0%}: ~{total_visits} total visits across all years/areas")

## Summary

| Feature | Function | Purpose |
|---------|----------|----------|
| Cloud data retrieval | `create_cloud_data_array_with_limit()` | MODIS daily cloud fraction from GEE |
| Visit simulation | `simulate_visits()` | Realistic scheduling with constraints |
| Visualization | `plot_yearly_cloud_fraction_heatmaps_with_visits()` | Heatmap with visit/rest markers |

### Tips for Campaign Planning

- **Cloud threshold 0.25** is a good starting point for imaging spectroscopy
- **Rest days** are important for crew safety and aircraft maintenance
- **Weekend exclusion** reduces available days by ~30%
- Query **multiple years** to understand climatological variability
- Use results to set realistic campaign durations and deployment windows